# B2-024 — Practice p08

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** core · **Time budget:** 50 minutes  
**Concepts:** open-ended-model-evaluation

## Task

Implement percentile-bootstrap confidence intervals for a metric and for a paired difference.

**Contract.**

* `bootstrap_ci(metric_fn, y_true, y_pred, n_boot, alpha, generator)` returns a tuple of three Python floats `(point, low, high)`. `point = float(metric_fn(y_true, y_pred))`. With `n = y_true.shape[0]`, draw **once** `idx = torch.randint(0, n, (n_boot, n), generator=generator)`; for each row `b`, the replicate statistic is `float(metric_fn(y_true[idx[b]], y_pred[idx[b]]))` (labels and predictions resampled **together**). `(low, high)` are `torch.quantile` (default linear interpolation) of the `float64` tensor of the `n_boot` replicate statistics at `alpha / 2` and `1 - alpha / 2`.
* `paired_bootstrap_diff(metric_fn, y_true, pred_a, pred_b, n_boot, alpha, generator)` returns `(point, low, high)` for the difference `metric(a) - metric(b)`: the point is `float(metric_fn(y_true, pred_a)) - float(metric_fn(y_true, pred_b))`; one `idx` drawn as above is shared by both models, and each replicate is the difference of the two resampled metrics; same quantiles.

**Banned:** NumPy or SciPy random generators, `torch.manual_seed` inside either function, and any resampling that does not use the passed `generator`.

**Probes.** With the supplied literals, `accuracy = lambda t, p: (t == p).double().mean()`, `n_boot = 2000`, `alpha = 0.1`, and a **fresh** `torch.Generator().manual_seed(7)` for each call:

1. `bootstrap_ci(accuracy, y_true, pred_a, ...)` is `(0.75, 0.5, 0.9166666666666666)`;
2. `bootstrap_ci(accuracy, y_true, pred_b, ...)` is `(0.5, 0.25, 0.75)`;
3. `paired_bootstrap_diff(accuracy, y_true, pred_a, pred_b, ...)` is `(0.25, 0.08333333333333326, 0.49999999999999994)`;

all with `math.isclose(..., rel_tol=0, abs_tol=1e-9)` on each component.

4. All three return Python `float`s, and `low <= point <= high` holds in probes 1–2.
5. Reproducibility: two calls with fresh generators seeded 7 give identical tuples.
6. Wrong variant: write `wrong_bootstrap_ci` that resamples only the predictions (`metric_fn(y_true, y_pred[idx[b]])`). With a fresh generator seeded 7 on `pred_a` its interval is `[0.08333333333333333, 0.5833333333333334]`; assert that it differs from probe 1's, and explain in two sentences what it actually measures.
7. Explain in three sentences why the paired interval in probe 3 excludes 0 even though the separate intervals of probes 1 and 2 overlap.

**Required answer-check assertions.** Probes 1–6.

In [ ]:
import math
import torch

y_true = torch.tensor([0, 1, 2, 1, 0, 2, 2, 1, 0, 1, 2, 0])
pred_a = torch.tensor([0, 1, 2, 2, 0, 2, 1, 1, 0, 1, 2, 1])
pred_b = torch.tensor([0, 2, 2, 2, 1, 2, 1, 1, 0, 0, 2, 1])
N_BOOT = 2000
ALPHA = 0.1
ABS_TOL = 1e-9

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.